# Costruzione del knowledge graph

Tabelle integrate → triple RDF → Neo4j. 

**Prerequisiti:** `pip install rdflib neo4j`; `data/processed/`; `data/raw/dataset_1/worldcup.db`; l'ontologia `ontology/wc-ontology.ttl`; Neo4j 5.26 con il plugin n10s 5.26.0 avviato (celle 29–32).

| Celle | Contenuto |
|---|---|
| 1–2 | Setup e funzioni comuni alle tabelle |
| 3–7 | Participation e archi hasManager |
| 8–11 | Tabelle a livello di torneo |
| 12 | Persone e premi |
| 13–15 | Stadi e partite |
| 16–19 | Club e marche |
| 20–21 | Convocazioni |
| 22–23 | Presenze ed eventi |
| 24 | Immagini |
| 25–28 | Triple RDF |
| 29–32 | Import in Neo4j |

## Setup

In [ ]:
# [Cella 1] Setup: librerie, percorsi, connessione al database del dataset 1, dati del preprocessing
import itertools
import re
import sqlite3
from getpass import getpass
from pathlib import Path
from urllib.parse import unquote

import numpy as np
import pandas as pd
from neo4j import GraphDatabase
from rdflib import Graph, Literal, Namespace, URIRef, OWL, RDF, RDFS, XSD
from rapidfuzz import fuzz
from rdflib.namespace import FOAF, SKOS
from unidecode import unidecode

# Radice del progetto
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB = ROOT / "data" / "raw" / "dataset_1" / "worldcup.db"
OUT = ROOT / "data" / "processed"          # tabelle integrate (Parquet)
KG = ROOT / "data" / "kg"                  # triple RDF (un file Turtle per tabella)
assert DB.exists() and OUT.exists(), "Controlla ROOT: " + str(ROOT)
KG.mkdir(parents=True, exist_ok=True)

# Database del dataset 1 in sola lettura 
con = sqlite3.connect(DB.as_uri() + "?mode=ro", uri=True)

# Dati puliti del preprocessing 
st = pd.read_parquet(OUT / "dataset_2_clean.parquet")      # Dataset 2: classifiche finali 1930–2022
pl = pd.read_parquet(OUT / "dataset_3_clean.parquet")     # Dataset 3: giocatori 2022 con player_id
imgs = pd.read_csv(OUT / "images_index.csv")            # indice delle immagini con player_id

print("st", st.shape, "| pl", pl.shape, "| imgs", imgs.shape)
assert st.shape == (489, 13) and pl.shape == (814, 22) and imgs.shape == (41510, 13)
assert pl["player_id"].notna().all()

st (489, 13) | pl (814, 22) | imgs (41510, 13)


In [ ]:
# [Cella 2] Funzioni di supporto per le tabelle
# Successioni riconosciute dalla FIFA: predecessore → successore
PREDECESSORI = {"T-86": "T-31", "T-72": "T-62", "T-87": "T-67", "T-67": "T-66", "T-21": "T-20"}
# Segnaposto del dataset 1 per "nessun valore" (squadre: "not applicable"; persone anche "not available")
MARCATORI = ["not applicable", "not available"]
# Caratteri vietati negli IRI: nei titoli DBpedia vanno ricodificati come %XX (es. " → %22)
VIETATI = '<>"{}|\\^` '

def data(s):
    # Date dei tornei e delle fasi: interi, giorni dal 1970-01-01 (-14417 → 1930-07-13)
    return pd.to_datetime(s, unit="D").dt.date

def booleano(s):
    # I booleani sono 0/1: ci si ferma se compare un valore diverso invece di convertirlo male
    assert s.isin([0, 1]).all(), f"{s.name}: valori diversi da 0/1"
    return s.astype(bool)

def pulisci(s):
    # Segnaposto (e stringa vuota) → valore mancante, quindi nessuna tripla
    return s.mask(s.isin(MARCATORI + [""]))

def lettera_girone(s):
    # "Group A" → "A", "Group c" → "C": la lettera entra nell'URI del girone
    return s.str.replace("Group ", "", regex=False).str.upper()

def dbpedia(url):
    # La risorsa DBpedia ha lo stesso titolo della pagina Wikipedia: si decodifica 
    # e si ricodificano solo i caratteri vietati negli IRI
    if url in MARCATORI:
        return None
    titolo = unquote(url.rsplit("/wiki/", 1)[1])
    titolo = "".join(f"%{ord(c):02X}" if c in VIETATI else c for c in titolo)
    return "http://dbpedia.org/resource/" + titolo

def collegamenti(df, col_link, id_col):
    # Tre colonne di link esterni. Se un link è condiviso nella stessa tabella,
    # il predecessore temporale riceve solo rdfs:seeAlso (sameAs e isPrimaryTopicOf implicano identità)
    link = df[col_link].where(~df[col_link].isin(MARCATORI))
    condiviso = link.notna() & link.duplicated(keep=False)      # keep=False: segna tutte le copie
    usa_seealso = condiviso & df[id_col].isin(PREDECESSORI.keys())
    df["sameAs"] = link.map(dbpedia, na_action="ignore").where(~usa_seealso)
    df["isPrimaryTopicOf"] = link.where(~usa_seealso)
    df["seeAlso"] = link.where(usa_seealso)
    return df

def slug_nome(nome):
    # Regola degli URI per club e marche: "New Balance" → "new-balance"
    return re.sub(r"[^a-z0-9]+", "-", unidecode(str(nome)).lower()).strip("-")

def manca(figli, padri, chiave):
    # Quante chiavi dei "figli" non trovano un "padre": in RDF un arco verso un URI
    # inesistente non dà errore, crea un nodo vuoto in Neo4j
    k = figli[chiave].drop_duplicates().merge(padri[chiave].drop_duplicates(), how="left", indicator=True)
    return int((k["_merge"] == "left_only").sum())

## Tabella Participation e archi hasManager
Una riga per coppia (torneo, squadra)

In [ ]:
# [Cella 3] Participation: base dal dataset 1 (qualified_teams + teams + tournaments)
# Una riga per squadra e torneo: la chiave (tournament_id, team_id) diventerà l'URI
part = pd.read_sql("""
    SELECT q.tournament_id, q.team_id, t.team_name,
           tr.tournament_name, tr.year, q.performance
    FROM qualified_teams AS q
    JOIN teams       AS t  ON t.team_id = q.team_id
    JOIN tournaments AS tr ON tr.tournament_id = q.tournament_id
""", con)
# Genere dal nome del torneo: serve per l'URI della nazionale (nationalteam/{team_id}/{gender})
part["gender"] = np.where(part["tournament_name"].str.contains("Women"), "women", "men")
# I tornei femminili scrivono la fase al singolare: si uniforma al plurale
part["performance"] = part["performance"].replace({"quarter-final": "quarter-finals"})
# Etichetta leggibile (rdfs:label), utile anche per la ricerca testuale
part["label"] = part["team_name"] + " at the " + part["tournament_name"]
CHIAVE = ["tournament_id", "team_id"]
print("base:", part.shape)

base: (625, 8)


In [ ]:
# [Cella 4] Participation: isHost (host_countries) e finalPosition (tournament_standings)
# indicator dice se la riga ha trovato corrispondenza ("both")
host = pd.read_sql("SELECT tournament_id, team_id FROM host_countries", con)
part = part.merge(host, on=CHIAVE, how="left", indicator="fonte_host", validate="one_to_one")
part["isHost"] = part.pop("fonte_host").eq("both")

# finalPosition esiste solo per le prime quattro; Int64 ammette i mancanti senza diventare decimale
podio = pd.read_sql("SELECT tournament_id, team_id, position AS finalPosition FROM tournament_standings", con)
part = part.merge(podio, on=CHIAVE, how="left", validate="one_to_one")
part["finalPosition"] = part["finalPosition"].astype("Int64")

In [ ]:
# [Cella 5] Participation: finalRank (dataset 2, solo tornei maschili)
# Il dataset 2 non ha team_id: si aggancia per anno + nome uniformato (alias del notebook 01);
# gender = "men" impedisce agganci a tornei femminili
dataset_2 = st[["year", "team_name", "Position"]].rename(columns={"Position": "finalRank"}).assign(gender="men")
part = part.merge(dataset_2, on=["year", "team_name", "gender"], how="left", validate="one_to_one")
part["finalRank"] = part["finalRank"].astype("Int64")

In [ ]:
# [Cella 6] Participation: fifaRanking e kitSponsor (dataset 3, solo 2022)
# Valori ripetuti su ogni giocatore ma della squadra: groupby 814 → 32; "first" salta i mancanti
rh = (pl.groupby("team_name", as_index=False)
        .agg(fifaRanking=("FIFA Ranking", "first"), kitSponsor=("National Team Kit Sponsor", "first"))
        .assign(tournament_id="WC-2022"))
part = part.merge(rh, on=["tournament_id", "team_name"], how="left", validate="one_to_one")
part["fifaRanking"] = part["fifaRanking"].astype("Int64")

In [ ]:
# [Cella 7] Archi hasManager e salvataggio delle due tabelle
# Tabella di archi a parte, perché 12 partecipazioni hanno due allenatori
mgr = pd.read_sql("SELECT tournament_id, team_id, manager_id FROM manager_appointments", con)

# year e team_name restano solo per i controlli: non diventano triple
part = part[["tournament_id", "team_id", "gender", "year", "team_name", "label", "performance",
             "isHost", "finalPosition", "finalRank", "fifaRanking", "kitSponsor"]]
part.to_parquet(OUT / "participation.parquet", index=False)
mgr.to_parquet(OUT / "participation_manager.parquet", index=False)
print("salvati:", part.shape, mgr.shape)

salvati: (625, 12) (637, 3)


## Tabelle su tornei e federazioni
Tournament, Stage, Group, GroupStanding, Confederation, Federation, archi predecessorOf, NationalTeam (tutte dal dataset 1). 

In [ ]:
# [Cella 8] Tournament, Stage, Group
tour = pd.read_sql("""
    SELECT tournament_id, tournament_name AS label, year, start_date, end_date,
           count_teams AS countTeams
    FROM tournaments""", con)
tour["gender"] = np.where(tour["label"].str.contains("Women"), "women", "men")
tour["startDate"] = data(tour.pop("start_date"))       # pop: toglie la colonna originale mentre la converte
tour["endDate"] = data(tour.pop("end_date"))

# Stage: il nome viene da tournament_stages, che usa nomi uniformi
stage = pd.read_sql("""
    SELECT s.tournament_id, s.stage_number AS stageNumber, s.stage_name AS stageName,
           s.group_stage, s.knockout_stage, s.start_date, s.end_date, t.tournament_name
    FROM tournament_stages AS s
    JOIN tournaments AS t ON t.tournament_id = s.tournament_id""", con)
stage["isGroupStage"] = booleano(stage.pop("group_stage"))
stage["isKnockoutStage"] = booleano(stage.pop("knockout_stage"))
stage["startDate"] = data(stage.pop("start_date"))
stage["endDate"] = data(stage.pop("end_date"))
stage["label"] = stage.pop("tournament_name") + ", " + stage["stageName"]

# Group: collegato alla fase con il NUMERO, non con il nome (groups.stage_name è incoerente)
group = pd.read_sql("""
    SELECT g.tournament_id, g.stage_number AS stageNumber, g.group_name,
           g.count_teams AS countTeams, t.tournament_name
    FROM groups AS g
    JOIN tournaments AS t ON t.tournament_id = g.tournament_id""", con)
group["groupLetter"] = lettera_girone(group.pop("group_name"))
group = group.merge(stage[["tournament_id", "stageNumber", "stageName"]],
                    on=["tournament_id", "stageNumber"], how="left", validate="many_to_one")
# L'etichetta include la fase: nel 1982 esiste un "Group A" in due fasi
group["label"] = ("Group " + group["groupLetter"] + " (" + group.pop("stageName") + "), "
                  + group.pop("tournament_name"))

print("tour", tour.shape, "| stage", stage.shape, "| group", group.shape)

tour (30, 7) | stage (155, 8) | group (159, 5)


In [ ]:
# [Cella 9] GroupStanding: classifica di una squadra nel suo girone
gst = pd.read_sql("""
    SELECT tournament_id, stage_number AS stageNumber, group_name, team_id,
           position AS groupPosition, points, advanced
    FROM group_standings""", con)
gst["groupLetter"] = lettera_girone(gst.pop("group_name"))
gst["advanced"] = booleano(gst["advanced"])
print("gst", gst.shape)

gst (626, 7)


In [ ]:
# [Cella 10] Confederation, Federation, archi predecessorOf, NationalTeam
conf = pd.read_sql("""
    SELECT confederation_id, confederation_name AS label, confederation_code AS confederationCode,
           confederation_wikipedia_link AS link
    FROM confederations""", con)
conf = collegamenti(conf, "link", "confederation_id").drop(columns="link")

teams = pd.read_sql("SELECT * FROM teams", con)

# Federation: label = paese, altLabel = nome ufficiale; confederation_id → memberOf
fed = teams[["team_id", "team_name", "federation_name", "team_code", "region_name",
             "confederation_id", "federation_wikipedia_link"]].rename(columns={
    "team_name": "label", "federation_name": "altLabel", "team_code": "teamCode",
    "region_name": "regionName", "federation_wikipedia_link": "link"})
fed = collegamenti(fed, "link", "team_id").drop(columns="link")

# Archi predecessorOf
nome_squadra = teams.set_index("team_id")["team_name"]
pred = pd.DataFrame(list(PREDECESSORI.items()), columns=["team_id", "successor_id"])
pred["team_name"] = pred["team_id"].map(nome_squadra)
pred["successor_name"] = pred["successor_id"].map(nome_squadra)
assert pred.notna().all().all(), "un id di PREDECESSORI non esiste in teams"

# NationalTeam: fino a due nazionali per federazione, ognuna con il proprio link Wikipedia
nt = []
for genere, flag, col in [("men", "mens_team", "mens_team_wikipedia_link"),
                          ("women", "womens_team", "womens_team_wikipedia_link")]:
    sel = teams.loc[booleano(teams[flag]), ["team_id", "team_name", col]].rename(columns={col: "link"})
    sel["gender"] = genere
    sel["label"] = sel.pop("team_name") + " (" + genere + ")"
    nt.append(collegamenti(sel, "link", "team_id").drop(columns="link"))
nt = pd.concat(nt, ignore_index=True)
print("conf", conf.shape, "| fed", fed.shape, "| pred", pred.shape, "| nt", nt.shape)

conf (6, 6) | fed (88, 9) | pred (5, 4) | nt (121, 6)


In [ ]:
# [Cella 11] Salvataggio delle tabelle
TABELLE = {"tournament": tour, "stage": stage, "group": group, "group_standing": gst,
           "confederation": conf, "federation": fed, "federation_predecessor": pred,
           "national_team": nt}
for nome, df in TABELLE.items():
    df.to_parquet(OUT / f"{nome}.parquet", index=False)
    print(f"{nome:23} {df.shape}")

tournament              (30, 7)
stage                   (155, 8)
group                   (159, 5)
group_standing          (626, 7)
confederation           (6, 6)
federation              (88, 9)
federation_predecessor  (5, 4)
national_team           (121, 6)


## Persone e premi
Player, Manager, Referee, Award dal dataset 1.

In [ ]:
# [Cella 12] Player, Manager, Referee, Award
player = pd.read_sql("""
    SELECT player_id, given_name AS givenName, family_name AS familyName,
           birth_date AS birthDate, female, player_wikipedia_link AS link
    FROM players""", con)
manager = pd.read_sql("""
    SELECT manager_id, given_name AS givenName, family_name AS familyName,
           female, country_name AS countryName, manager_wikipedia_link AS link
    FROM managers""", con)
referee = pd.read_sql("""
    SELECT referee_id, given_name AS givenName, family_name AS familyName,
           female, country_name AS countryName, confederation_id, referee_wikipedia_link AS link
    FROM referees""", con)

# Stesse operazioni sulle tre tabelle di persone
for df, chiave in [(player, "player_id"), (manager, "manager_id"), (referee, "referee_id")]:
    for c in df.columns.intersection(["givenName", "familyName", "link", "birthDate"]):
        df[c] = pulisci(df[c])
    # Nome e cognome; chi è noto con un solo nome ha il nome mancante ("Pelé")
    df["label"] = (df["givenName"].fillna("") + " " + df["familyName"]).str.strip()
    df["isFemale"] = booleano(df.pop("female"))
    collegamenti(df, "link", chiave)     # stessa persona in due ruoli → stesso sameAs (corretto)
    df.drop(columns="link", inplace=True)
# Le date dei giocatori sono testo ISO: con il formato esplicito una data malformata dà errore
player["birthDate"] = pd.to_datetime(player["birthDate"], format="%Y-%m-%d")

award = pd.read_sql("""
    SELECT award_id, award_name AS label, award_description AS awardDescription,
           year_introduced AS yearIntroduced
    FROM awards""", con)

for nome, df in {"player": player, "manager": manager, "referee": referee, "award": award}.items():
    df.to_parquet(OUT / f"{nome}.parquet", index=False)
    print(f"{nome:8} {df.shape}")

player   (10401, 9)
manager  (475, 9)
referee  (493, 10)
award    (8, 4)


## Stadi e partite
Stadium e Match dal dataset 1.

In [ ]:
# [Cella 13] Stadium: stadi (come nella fonte)
stadium = pd.read_sql("""
    SELECT stadium_id, stadium_name AS label, city_name AS cityName, country_name AS countryName,
           stadium_capacity AS capacity, stadium_wikipedia_link AS link
    FROM stadiums""", con)
# Sei stadi compaiono due volte con id diversi (1994/1999 negli Stati Uniti, due refusi nel 1991):
# ricevono lo stesso sameAs, che in Neo4j li collega alla stessa risorsa DBpedia
collegamenti(stadium, "link", "stadium_id")
stadium = stadium.drop(columns="link")
print("stadi:", len(stadium), "| con la pagina Wikipedia di un altro stadio:", stadium["sameAs"].duplicated(keep=False).sum())

stadi: 240 | con la pagina Wikipedia di un altro stadio: 12


In [ ]:
# [Cella 14] Match: partite (dati della partita, arbitro, stadio)
# referee_appearances ha un arbitro per partita: LEFT JOIN per non perdere partite se mancasse
match = pd.read_sql("""
    SELECT m.match_id, m.tournament_id, m.match_name AS label, m.stage_name, m.group_stage,
           m.match_date, m.stadium_id, m.home_team_id, m.away_team_id,
           m.home_team_score AS homeScore, m.away_team_score AS awayScore,
           m.extra_time, m.penalty_shootout, m.replay,
           m.home_team_score_penalties AS homePenalties, m.away_team_score_penalties AS awayPenalties,
           r.referee_id
    FROM matches AS m
    LEFT JOIN referee_appearances AS r ON r.match_id = m.match_id""", con)
# Qui la data è testo ("-14417.0"): prima numero, poi giorni dal 1970-01-01
match["matchDate"] = pd.to_datetime(pd.to_numeric(match.pop("match_date")), unit="D")
match["extraTime"] = booleano(match.pop("extra_time"))
match["penaltyShootout"] = booleano(match.pop("penalty_shootout"))
match["isReplay"] = booleano(match.pop("replay"))        # ripetizione di una partita finita in parità (1934, 1938)
# Senza serie di rigori il dataset 1 scrive 0-0: è un "non applicabile", non un risultato → mancante
for c in ["homePenalties", "awayPenalties"]:
    match[c] = match[c].astype("Int64").where(match["penaltyShootout"])
print("partite:", match.shape)

partite: (1248, 17)


In [ ]:
# [Cella 15] Match: fase e girone di ogni partita (girone ricavato dalle due squadre) e salvataggio
gironi = pd.read_parquet(OUT / "group_standing.parquet")[["tournament_id", "stageNumber", "groupLetter", "team_id"]]
fasi = pd.read_parquet(OUT / "stage.parquet")[["tournament_id", "stageNumber", "stageName", "startDate", "endDate"]]

# 1) Partite a gironi: il girone è quello che, nello stesso torneo, contiene ENTRAMBE le squadre.
#    Primo merge: gironi della squadra di casa; secondo merge: tra questi, quelli che contengono anche
#    la squadra in trasferta (stessa fase e stessa lettera). Il nome del girone scritto in matches non si usa.
a_gironi = match[match["group_stage"] == 1]
cand = (a_gironi[["match_id", "tournament_id", "home_team_id", "away_team_id", "matchDate"]]
        .merge(gironi.rename(columns={"team_id": "home_team_id"}), on=["tournament_id", "home_team_id"])
        .merge(gironi.rename(columns={"team_id": "away_team_id"}),
               on=["tournament_id", "stageNumber", "groupLetter", "away_team_id"])
        .merge(fasi, on=["tournament_id", "stageNumber"]))
# Sicurezza: se due squadre fossero nello stesso girone in due fasi diverse, decide la data della partita
giorno = cand["matchDate"].dt.date
data_ok = (giorno >= cand["startDate"]) & (giorno <= cand["endDate"])
print("candidati:", len(cand), "| fuori dalle date della fase:", (~data_ok).sum())
cand = cand[data_ok]
per_partita = cand.groupby("match_id").size()
print("partite a gironi:", len(a_gironi), "| con 1 girone:", (per_partita == 1).sum(),
      "| con più gironi:", (per_partita > 1).sum(),
      "| senza girone:", len(set(a_gironi["match_id"]) - set(per_partita.index)))
assert (per_partita == 1).all() and len(per_partita) == len(a_gironi)

# 2) Partite a eliminazione: fase per nome, dopo aver uniformato il singolare dei tornei femminili
elim = match.loc[match["group_stage"] == 0, ["match_id", "tournament_id", "stage_name"]]
elim = elim.assign(stageName=elim["stage_name"].replace({"quarter-final": "quarter-finals", "semi-final": "semi-finals"}))
elim = elim.merge(fasi[["tournament_id", "stageName", "stageNumber"]],
                  on=["tournament_id", "stageName"], how="left", validate="many_to_one")
print("partite a eliminazione:", len(elim), "| senza fase:", elim["stageNumber"].isna().sum())
assert elim["stageNumber"].notna().all()

# 3) Fase (e girone) di ogni partita; via le colonne del dataset 1 non più necessarie
fase_partita = pd.concat([cand[["match_id", "stageNumber", "groupLetter"]],
                          elim[["match_id", "stageNumber"]]], ignore_index=True)
match = (match.merge(fase_partita, on="match_id", how="left", validate="one_to_one")
              .drop(columns=["stage_name", "group_stage"]))
match["stageNumber"] = match["stageNumber"].astype("Int64")

stadium.to_parquet(OUT / "stadium.parquet", index=False)
match.to_parquet(OUT / "match.parquet", index=False)
print("salvati:", stadium.shape, match.shape)

candidati: 922 | fuori dalle date della fase: 0
partite a gironi: 922 | con 1 girone: 922 | con più gironi: 0 | senza girone: 0
partite a eliminazione: 326 | senza fase: 0
salvati: (240, 8) (1248, 17)


## Club e marche
Club (dal dataset 3, colonna `Club`) e marche (sponsor tecnico delle nazionali e scarpe dei giocatori). 

In [ ]:
# [Cella 16] Club: inventario dei nomi e chiave normalizzata
# quanti nomi distinti e quanto spesso compaiono
club_righe = pl.dropna(subset=["Club"])[["Player Name", "team_name", "Club"]]
print("righe con club:", len(club_righe), "| senza club:", pl["Club"].isna().sum(),
      "| nomi distinti:", club_righe["Club"].nunique())

# chiave normalizzata = minuscolo, senza accenti e punteggiatura, senza le sigle della
# forma societaria (FC, CF, SC, AC, AS, AFC). Nomi con la stessa chiave = stesso club, per regola
SIGLE = {"fc", "cf", "sc", "ac", "as", "afc"}

def chiave_club(nome):
    parole = re.sub(r"[^a-z0-9 ]", " ", unidecode(nome).lower()).split()
    return " ".join(p for p in parole if p not in SIGLE)

nomi_club = pd.DataFrame({"Club": sorted(club_righe["Club"].unique())})
nomi_club["chiave"] = nomi_club["Club"].map(chiave_club)
per_chiave = nomi_club.groupby("chiave")["Club"].agg(list)
print("chiavi distinte:", len(per_chiave), "| chiavi con più grafie (unite per regola):", (per_chiave.str.len() > 1).sum())
print(per_chiave[per_chiave.str.len() > 1].to_string())

righe con club: 812 | senza club: 2 | nomi distinti: 316
chiavi distinte: 304 | chiavi con più grafie (unite per regola): 12
chiave
al ahli                                                    [Al Ahli, Al-Ahli]
al sadd                                                    [Al Sadd, Al-Sadd]
al shabab                                              [Al Shabab, Al-Shabab]
al wehda                                                 [Al Wehda, Al-Wehda]
augsburg                                              [Augsburg, FC Augsburg]
barcelona                                           [Barcelona, FC Barcelona]
borussia monchengladbach    [Borussia Monchengladbach, Borussia Mönchengla...
braga                                                       [Braga, SC Braga]
copenhagen                                        [Copenhagen, FC Copenhagen]
freiburg                                              [Freiburg, SC Freiburg]
monaco                                                    [AS Monaco, Monaco]
seattle so

In [ ]:
# [Cella 17] Club, coppie candidate (stesso club scritto in modo diverso?)
# Si confrontano tutte le coppie di chiavi; una coppia è candidata se vale almeno uno di tre segnali:
#   contenuto     → le parole della chiave corta sono tutte nella lunga ("leicester" / "leicester city")
#   simile        → somiglianza tra le stringhe ≥ 85 su 100 (refusi: "sepahan" / "sepahann")
#   abbreviazione → stessa iniziale e le lettere della corta compaiono, in ordine, nella lunga ("man utd")
# I segnali servono a non perdere varianti (richiamo alto); la decisione resta al passo 4
def sottosequenza(corto, lungo):
    resto = iter(lungo)
    return all(lettera in resto for lettera in corto)

coppie = []
for a, b in itertools.combinations(sorted(per_chiave.index), 2):
    corto, lungo = sorted([a, b], key=len)
    c, l = corto.replace(" ", ""), lungo.replace(" ", "")
    contenuto = set(corto.split()) < set(lungo.split())
    simile = fuzz.ratio(a, b) >= 85
    abbreviazione = not contenuto and len(c) >= 3 and c[0] == l[0] and sottosequenza(c, l)
    segnali = [n for n, v in [("contenuto", contenuto), ("simile", simile), ("abbreviazione", abbreviazione)] if v]
    if segnali:
        coppie.append({"corto": corto, "lungo": lungo, "somiglianza": round(fuzz.ratio(a, b)),
                       "segnali": "+".join(segnali)})
candidate = pd.DataFrame(coppie)

# Prova per la decisione: i giocatori registrati con ciascun nome (nome e nazionale)
club_righe = club_righe.assign(chiave=club_righe["Club"].map(chiave_club))
def giocatori(chiave):
    d = club_righe[club_righe["chiave"] == chiave]
    return f"{len(d)}: " + "; ".join(d["Player Name"] + " (" + d["team_name"] + ")")

n = len(per_chiave)
print("coppie confrontate:", n * (n - 1) // 2, "| candidate:", len(candidate))
print(candidate["segnali"].value_counts().to_dict())
for r in candidate.itertuples():
    print(f"\n{r.corto!r} / {r.lungo!r}  [{r.segnali}, {r.somiglianza}]")
    print("   ", giocatori(r.corto)[:150])
    print("   ", giocatori(r.lungo)[:150])

coppie confrontate: 46056 | candidate: 32
{'contenuto': 14, 'abbreviazione': 13, 'simile': 4, 'simile+abbreviazione': 1}

'al ahli' / 'al ahly'  [simile, 86]
    4: Hossein Kanaani (Iran); Shojae Khalilzadeh (Iran); Mohammed Al Yami (Saudi Arabia); Haitham Asiri (Saudi Arabia)
    1: Ali Maaloul (Tunisia)

'al ahli' / 'shabab al ahli'  [contenuto, 67]
    4: Hossein Kanaani (Iran); Shojae Khalilzadeh (Iran); Mohammed Al Yami (Saudi Arabia); Haitham Asiri (Saudi Arabia)
    1: Ahmad Nourollahi (Iran)

'al shabab' / 'shabab al ahli'  [contenuto, 52]
    4: Grzegorz Krychowiak (Poland); Kim Seung-gyu (South Korea); Hassan Tambakti (Saudi Arabia); Nawaf Alabid (Saudi Arabia)
    1: Ahmad Nourollahi (Iran)

'al tai' / 'atlanta united'  [abbreviazione, 50]
    1: Collins Fai (Cameroon)
    1: Alejandro Gomez (Argentina)

'al tai' / 'atletico madrid'  [abbreviazione, 48]
    1: Collins Fai (Cameroon)
    12: Nahuel Molina (Argentina); Rodrigo De Paul (Argentina); Angel Correa (Argentina); Ant

In [ ]:
# [Cella 18] Club: decisione su ogni coppia candidata e tabella dei club
# ALIAS: stesso club → chiave della variante : chiave del nome da tenere (forma completa o corretta)
ALIAS = {
    "leicester": "leicester city",
    "man utd": "manchester united",
    "montreal impact": "montreal",              # CF Montréal è il nome attuale (dal 2021) del Montreal Impact
    "river": "river plate",
    "salzburg": "red bull salzburg",
    "sporting": "sporting cp",
    "tottenham": "tottenham hotspur",
    "valladolid": "real valladolid",
    "verona": "hellas verona",
    "west ham": "west ham united",
    "wolves": "wolverhampton wanderers",
    "sepahann": "sepahan",                      # refuso
    "aprissa": "saprissa",                      # refuso (Deportivo Saprissa, Costa Rica)
}
# DIVERSI: club diversi con nomi simili, da NON unire; la prova sono i giocatori
DIVERSI = {
    ("al ahli", "al ahly"),                    
    ("al ahli", "shabab al ahli"),              
    ("al shabab", "shabab al ahli"),
    ("almeria", "america"),                     
    ("angers", "rangers"),                      
    ("hearts", "hearts of oak"),                
    ("independiente", "independiente del valle"),   
    ("inter", "inter miami"),                   
    ("al tai", "atlanta united"), ("al tai", "atletico madrid"), ("bari", "bayern munich"),
    ("braga", "borussia monchengladbach"), ("caen", "copenhagen"), ("caen", "cs sfaxien"),
    ("lens", "leganes"), ("leon", "lech poznan"), ("lugo", "lugano"), ("paok", "panaitolikos"),
    ("porto", "portsmouth"),
}
# Ogni coppia candidata deve essere decisa, e ogni decisione deve riguardare una coppia candidata
decise = {tuple(sorted(p)) for p in ALIAS.items()} | {tuple(sorted(p)) for p in DIVERSI}
trovate = {tuple(sorted(p)) for p in candidate[["corto", "lungo"]].itertuples(index=False)}
assert decise == trovate, f"non decise: {trovate - decise} | non candidate: {decise - trovate}"

# Nome di ogni club: la grafia più frequente tra i nomi con la chiave da tenere (a pari merito la più lunga)
nomi_club["chiave_club"] = nomi_club["chiave"].replace(ALIAS)
frequenza = club_righe["Club"].value_counts()
nomi_club["frequenza"] = nomi_club["Club"].map(frequenza)
etichette = (nomi_club[nomi_club["chiave"] == nomi_club["chiave_club"]]
             .assign(lunghezza=lambda d: d["Club"].str.len())
             .sort_values(["frequenza", "lunghezza"], ascending=False)
             .drop_duplicates("chiave_club")
             .set_index("chiave_club")["Club"])
nomi_club["label"] = nomi_club["chiave_club"].map(etichette)
nomi_club["slug"] = nomi_club["label"].map(slug_nome)
nomi_club["metodo"] = np.select(
    [nomi_club["chiave"] != nomi_club["chiave_club"],
     nomi_club["Club"] != nomi_club["label"]],
    ["alias", "chiave"], default="identico")

club = nomi_club.drop_duplicates("slug")[["slug", "label"]].sort_values("slug").reset_index(drop=True)
assert club["label"].is_unique and nomi_club.groupby("chiave_club")["slug"].nunique().eq(1).all()
club.to_parquet(OUT / "club.parquet", index=False)
nomi_club[["Club", "chiave", "label", "slug", "metodo"]].to_parquet(OUT / "club_nomi.parquet", index=False)
print("nomi:", len(nomi_club), "→ club:", len(club), "|", nomi_club["metodo"].value_counts().to_dict())

nomi: 316 → club: 291 | {'identico': 291, 'alias': 13, 'chiave': 12}


In [ ]:
# [Cella 19] Brand: marche (sponsor tecnico delle nazionali 2022 e scarpe dei giocatori)
marche = pd.concat([pl["National Team Kit Sponsor"], pl["Brand Sponsor/Brand Used"]]).dropna()
brand = pd.DataFrame({"label": sorted(marche.unique())})
brand["slug"] = brand["label"].map(slug_nome)
# Stessa verifica delle varianti dei club: due grafie della stessa marca avrebbero lo stesso slug
assert brand["slug"].is_unique, brand[brand["slug"].duplicated(keep=False)]
brand.to_parquet(OUT / "brand.parquet", index=False)
print("marche:", len(brand), "|", ", ".join(brand["label"]))

marche: 15 | Adidas, Asics, Charly, Hummel, Kappa, Lotto, Majid, Marathon, Mizuno, New Balance, Nike, One All Sports, Puma, Umbro, Under Armour


## SquadMembership (convocazioni)
Un nodo per ogni giocatore nella rosa di una partecipazione (torneo + squadra). 

In [ ]:
# [Cella 20] SquadMembership: convocazioni dal dataset 1 (squads) e premi (award_winners)
sm = pd.read_sql("SELECT tournament_id, team_id, player_id, shirt_number, position_code FROM squads", con)
assert not sm.duplicated(["tournament_id", "player_id"]).any()   # un giocatore, una squadra per torneo
# Maglia 0 = sconosciuta (tornei 1930–1950) → mancante, quindi nessuna tripla (mondo aperto)
sm["shirtNumber"] = sm["shirt_number"].where(sm["shirt_number"] > 0).astype("Int64")
sm["position"] = sm["position_code"]          # GK, DF, MF, FW → wc:position_GK, ...

# Premi: tabella di soli archi, perché un giocatore può vincere più premi nello stesso torneo
premi = pd.read_sql("SELECT tournament_id, team_id, player_id, award_id FROM award_winners", con)
premi.to_parquet(OUT / "membership_award.parquet", index=False)

print("convocazioni:", len(sm), "| tornei:", sm["tournament_id"].nunique(),
      "| maglia sconosciuta:", sm["shirtNumber"].isna().sum(),
      sm.loc[sm["shirtNumber"].isna(), "tournament_id"].value_counts().sort_index().to_dict())
print("ruoli:", sm["position"].value_counts().to_dict())
print("premi:", len(premi), "| convocazioni premiate:", len(premi.drop_duplicates(["tournament_id", "player_id"])),
      "| premi senza convocazione:", manca(premi, sm, ["tournament_id", "team_id", "player_id"]))

convocazioni: 13843 | tornei: 30 | maglia sconosciuta: 1188 {'WC-1930': 245, 'WC-1934': 342, 'WC-1938': 320, 'WC-1950': 281}
ruoli: {'MF': 4269, 'DF': 4258, 'FW': 3545, 'GK': 1771}
premi: 200 | convocazioni premiate: 164 | premi senza convocazione: 0


In [ ]:
# [Cella 21] SquadMembership 2022: club, scarpe e statistiche (dataset 3), capitano (immagini); salvataggio
# Colonne del dataset 3 → proprietà dell'ontologia (prefisso "reported" per i dati che duplicano il dataset 1)
DATASET_3 = {"Appearances": "reportedAppearances", "Goals Scored": "reportedGoals",
           "Assists Provided": "assists", "Dribbles per 90": "dribblesPer90",
           "Interceptions per 90": "interceptionsPer90", "Tackles per 90": "tacklesPer90",
           "Total Duels Won per 90": "duelsWonPer90", "Save Percentage": "savePercentage",
           "Clean Sheet Rate": "cleanSheetRate"}
club_di = pd.read_parquet(OUT / "club_nomi.parquet").set_index("Club")["slug"]    # nome del dataset 3 → club (celle 16–18)
rh = (pl[["player_id", "Club", "Brand Sponsor/Brand Used", *DATASET_3]]
      .rename(columns=DATASET_3)
      .assign(tournament_id="WC-2022",
              club_slug=lambda d: d["Club"].map(club_di),
              bootBrand_slug=lambda d: d["Brand Sponsor/Brand Used"].map(slug_nome, na_action="ignore"))
      .drop(columns=["Club", "Brand Sponsor/Brand Used"]))
# Nuovo DataFrame (conv) invece di modificare sm: la cella si può rieseguire senza duplicare colonne
conv = sm.merge(rh, on=["tournament_id", "player_id"], how="left", validate="one_to_one", indicator="dataset_3")

# Capitano: vero per i 31 capitani delle cartelle; falso per gli altri convocati 2022 della stessa squadra;
# mancante dove la fonte non dice nulla (tornei precedenti, Stati Uniti: nessuna cartella "(captain)")
capitani = set(imgs.loc[imgs["captain"], "player_id"].dropna())
nel_2022 = conv["tournament_id"] == "WC-2022"
squadre_con_capitano = set(conv.loc[nel_2022 & conv["player_id"].isin(capitani), "team_id"])
conv["isCaptain"] = pd.Series(pd.NA, index=conv.index, dtype="boolean")
noto = nel_2022 & conv["team_id"].isin(squadre_con_capitano)
conv.loc[noto, "isCaptain"] = conv.loc[noto, "player_id"].isin(capitani)

conv.drop(columns="dataset_3").to_parquet(OUT / "squad_membership.parquet", index=False)
print("convocazioni:", len(conv), "| 2022:", nel_2022.sum(), "| con dati del dataset 3:", (conv["dataset_3"] == "both").sum(),
      "| righe del dataset 3 senza convocazione:", len(rh) - (conv["dataset_3"] == "both").sum())
print("club:", conv["club_slug"].notna().sum(), "| scarpe:", conv["bootBrand_slug"].notna().sum(),
      "| capitano: vero", (conv["isCaptain"] == True).sum(), "falso", (conv["isCaptain"] == False).sum(),
      "mancante", conv["isCaptain"].isna().sum())

convocazioni: 13843 | 2022: 831 | con dati del dataset 3: 814 | righe del dataset 3 senza convocazione: 0
club: 812 | scarpe: 812 | capitano: vero 31 falso 774 mancante 13038


## Presenze ed eventi delle partite
Dati del dataset 1: le presenze dei giocatori nelle partite (Appearance, dal 1970) e gli eventi (MatchEvent): gol, cartellini, sostituzioni, rigori delle serie finali.

In [ ]:
# [Cella 22] Appearance: presenze dei giocatori nelle partite (player_appearances)
# Una riga per giocatore e partita (dal 1970): diventerà un nodo Appearance
ap = pd.read_sql("""
    SELECT match_id, tournament_id, team_id, player_id, shirt_number, position_code, starter, substitute
    FROM player_appearances""", con)
assert not ap.duplicated(["match_id", "player_id"]).any()     # un giocatore compare una volta per partita
ap = ap.rename(columns={"shirt_number": "shirtNumber", "position_code": "position"})
ap["isStarter"] = booleano(ap.pop("starter"))
ap["isSubstitute"] = booleano(ap.pop("substitute"))
assert (ap["isStarter"] != ap["isSubstitute"]).all()          # titolare oppure entrato dalla panchina
assert (ap["shirtNumber"] > 0).all()                          # dal 1970 la maglia è sempre nota
ap.to_parquet(OUT / "appearance.parquet", index=False)
print("presenze:", len(ap), "| partite:", ap["match_id"].nunique(), "| tornei:", ap["tournament_id"].nunique(),
      "| titolari:", ap["isStarter"].sum(), "| dalla panchina:", ap["isSubstitute"].sum(),
      "| ruoli:", ap["position"].nunique())

presenze: 27432 | partite: 1015 | tornei: 21 | titolari: 22330 | dalla panchina: 5102 | ruoli: 21


In [ ]:
# [Cella 23] Eventi: gol, cartellini, sostituzioni, rigori delle serie finali
# Colonne del minuto, comuni a gol, cartellini e sostituzioni
MINUTO_COLONNE = {"minute_regulation": "minuteRegulation", "minute_stoppage": "minuteStoppage",
                  "match_period": "matchPeriod"}

def eventi(tabella, nome_file, id_col, booleani):
    # Una tabella del dataset 1 → eventi: identificativo unico, minuto, booleani 0/1 rinominati come nell'ontologia
    df = pd.read_sql(f"SELECT * FROM {tabella}", con).rename(columns=MINUTO_COLONNE)
    assert df[id_col].is_unique
    for col, nome in booleani.items():
        df[nome] = booleano(df.pop(col))
    df.to_parquet(OUT / f"{nome_file}.parquet", index=False)
    return df

# Nei gol team_id è la squadra a cui va il gol, player_team_id quella del giocatore (diverse negli autogol)
gol = eventi("goals", "goal", "goal_id", {"own_goal": "isOwnGoal", "penalty": "isPenalty"})
cartellini = eventi("bookings", "booking", "booking_id", {"yellow_card": "isYellowCard", "red_card": "isRedCard",
                                                "second_yellow_card": "isSecondYellow", "sending_off": "isSendingOff"})
sostituzioni = eventi("substitutions", "substitution", "substitution_id", {"going_off": "isGoingOff", "coming_on": "isComingOn"})
rigori = eventi("penalty_kicks", "penalty_kick", "penalty_kick_id", {"converted": "isConverted"})

print("gol:", len(gol), "| autogol:", gol["isOwnGoal"].sum(), "| su rigore:", gol["isPenalty"].sum())
print("cartellini:", len(cartellini), "| gialli:", cartellini["isYellowCard"].sum(),
      "| rossi:", cartellini["isRedCard"].sum(), "| espulsioni:", cartellini["isSendingOff"].sum())
print("sostituzioni:", len(sostituzioni), "| escono:", sostituzioni["isGoingOff"].sum(),
      "| entrano:", sostituzioni["isComingOn"].sum())
print("rigori delle serie:", len(rigori), "| segnati:", rigori["isConverted"].sum(),
      "| partite:", rigori["match_id"].nunique())
print("periodi:", gol["matchPeriod"].value_counts().to_dict())

gol: 3637 | autogol: 79 | su rigore: 292
cartellini: 3178 | gialli: 3092 | rossi: 102 | espulsioni: 172
sostituzioni: 10222 | escono: 5111 | entrano: 5111
rigori delle serie: 396 | segnati: 280 | partite: 43
periodi: {'second half': 1886, 'first half': 1507, 'second half, stoppage time': 121, 'extra time, first half': 40, 'first half, stoppage time': 40, 'extra time, second half': 39, 'extra time, second half, stoppage time': 3, 'extra time, first half, stoppage time': 1}


## Image (immagini dei giocatori)
Un nodo per file, collegato al giocatore raffigurato con `depicts`. Il giocatore viene dalla cartella del file, abbinata ai `player_id` nel notebook 01; l'etichetta non è verificata (il dataset è stato raccolto con ricerche web). La confidenza dell'etichetta (`labelConfidence`) si calcola con CLIP nel notebook 04 e si scrive direttamente in Neo4j.

In [ ]:
# [Cella 24] Image: un nodo per file, dalle cartelle abbinate ai giocatori
im = imgs.dropna(subset=["player_id"]).copy()       # escluse le 50 immagini di Martin Boyle, assente nel dataset 1
# Numero del file per l'URI image/{player_id}/{numero}: le ultime cifre prima dell'estensione
im["numero"] = im["file"].str.extract(r"(\d+)\.[A-Za-z]+$", expand=False)
assert im["numero"].notna().all() and not im.duplicated(["player_id", "numero"]).any()
im["filePath"] = im["path"].str.replace("\\", "/", regex=False)   # separatore "/" valido su ogni sistema
im = im.rename(columns={"format": "fileFormat", "bytes": "fileSize"})
# Contenuto duplicato: lo stesso file (stesso md5) nelle cartelle di giocatori diversi → almeno un'etichetta è sbagliata
giocatori_per_md5 = im.groupby("md5")["player_id"].nunique()
im["hasDuplicateContent"] = im["md5"].map(giocatori_per_md5) > 1

im = im[["player_id", "numero", "filePath", "md5", "fileFormat", "fileSize", "hasDuplicateContent", "team_name", "player_name"]]
im.to_parquet(OUT / "image.parquet", index=False)
print("immagini:", len(im), "| escluse (senza giocatore):", imgs["player_id"].isna().sum(),
      "| giocatori:", im["player_id"].nunique(), "| formati:", im["fileFormat"].value_counts().to_dict(),
      "| con contenuto duplicato:", im["hasDuplicateContent"].sum())

immagini: 41460 | escluse (senza giocatore): 50 | giocatori: 830 | formati: {'JPEG': 41338, 'PNG': 122} | con contenuto duplicato: 268


## Triple RDF
Una tabella → un file Turtle in `data/kg/`. Le regole sono nel dizionario `MAPPA` (cella 26).

In [ ]:
# [Cella 25] Ontologia, namespace e funzioni di traduzione tabella → triple
trovati = sorted(ROOT.glob("ontology/**/wc-ontology.ttl"))
assert len(trovati) == 1, f"Atteso un solo wc-ontology.ttl in ontology/, trovati: {trovati}"
# L'ontologia caricata con rdflib: da qui si controllano le proprietà e si leggono i tipi
onto = Graph().parse(trovati[0], format="turtle")

WC = Namespace("http://example.org/worldcup/ontology#")    # classi e proprietà: WC["year"]
R = Namespace("http://example.org/worldcup/resource/")     # nodi: R["tournament/WC-2022"]
SCHEMA = Namespace("https://schema.org/")

# Proprietà riusate da vocabolari esterni (non dichiarate nell'ontologia) e loro tipo
PROPRIETA_ESTERNE = {"label": RDFS.label, "altLabel": SKOS.altLabel,
                     "givenName": SCHEMA.givenName, "familyName": SCHEMA.familyName,
                     "birthDate": SCHEMA.birthDate}
TIPI_ESTERNI = {SCHEMA.birthDate: XSD.date}
LINK = {"sameAs": OWL.sameAs, "isPrimaryTopicOf": FOAF.isPrimaryTopicOf, "seeAlso": RDFS.seeAlso}

def predicato(nome):
    # Nome di colonna → proprietà RDF; le proprietà wc: devono esistere nell'ontologia
    if nome in PROPRIETA_ESTERNE:
        return PROPRIETA_ESTERNE[nome]
    p = WC[nome]
    assert (p, RDF.type, None) in onto, f"{nome} non è definita nell'ontologia"
    return p

def tipo_dato(p):
    # Il tipo del letterale si legge dal rdfs:range dell'ontologia (una sola fonte di verità)
    return onto.value(p, RDFS.range) or TIPI_ESTERNI.get(p, XSD.string)

def letterale(v, tipo):
    # Valore pandas → letterale RDF; int()/bool() convertono i tipi numpy
    if tipo == XSD.boolean:
        return Literal(bool(v))
    if tipo == XSD.integer:
        return Literal(int(v), datatype=XSD.integer)
    if tipo == XSD.decimal:
        return Literal(float(v), datatype=XSD.decimal)
    if tipo == XSD.date:
        return Literal(pd.Timestamp(v).date(), datatype=XSD.date)
    if tipo == XSD.gYear:
        return Literal(str(int(v)), datatype=XSD.gYear)
    return Literal(str(v))

def nodo(modello, riga):
    # Riempie il modello dell'URI con i valori della riga; se manca un campo → None (nessuna tripla)
    campi = re.findall(r"{(\w+)}", modello)
    if any(pd.isna(riga[c]) for c in campi):
        return None
    testo = modello.format(**{c: riga[c] for c in campi})
    # "wc:..." → nodo dell'ontologia (i ruoli, es. wc:position_GK); altrimenti risorsa del grafo
    return WC[testo[3:]] if testo.startswith("wc:") else R[testo]

def nuovo_grafo():
    g = Graph()
    for prefisso, ns in [("wc", WC), ("r", R), ("skos", SKOS), ("foaf", FOAF), ("schema", SCHEMA)]:
        g.bind(prefisso, ns)
    return g

def genera(df, classe=None, uri=None, dati=(), oggetti=None, link=False):
    # Righe → grafo rdflib; classe=None: tabella di soli archi
    g = nuovo_grafo()
    dati_p = [(c, predicato(c), tipo_dato(predicato(c))) for c in dati]
    oggetti_p = [(predicato(n), m) for n, m in (oggetti or {}).items()]
    for riga in df.to_dict("records"):
        s = nodo(uri, riga)
        if classe:
            g.add((s, RDF.type, WC[classe]))                       # tripla di tipo
        for col, p, tipo in dati_p:                                # proprietà dato
            if not pd.isna(riga[col]):
                g.add((s, p, letterale(riga[col], tipo)))
        for p, modello in oggetti_p:                               # proprietà oggetto
            o = nodo(modello, riga)
            if o is not None:
                g.add((s, p, o))
        if link:                                                   # link esterni
            for col, p in LINK.items():
                if not pd.isna(riga[col]):
                    g.add((s, p, URIRef(riga[col])))
    return g

print("ontologia:", len(onto), "triple")     # versione 0.3: 943

ontologia: 943 triple


In [ ]:
# [Cella 26] MAPPA: corrispondenza tabelle → ontologia
# Per ogni tabella: classe, modello dell'URI, colonne-dato, archi con il modello dell'URI di destinazione

DATASET_3_PROPRIETA = ["reportedAppearances", "reportedGoals", "assists", "dribblesPer90", "interceptionsPer90",
                     "tacklesPer90", "duelsWonPer90", "savePercentage", "cleanSheetRate"]
# Proprietà comuni agli eventi delle partite: minuto e archi verso partita, giocatore, squadra
MINUTO = ["minuteRegulation", "minuteStoppage", "matchPeriod"]
EVENTO = {"inMatch": "match/{match_id}", "player": "player/{player_id}",
          "participation": "participation/{tournament_id}/{team_id}"}
MAPPA = {
    # --- Partecipazioni e tabelle a livello di torneo
    "tournament": dict(classe="Tournament", uri="tournament/{tournament_id}",
                       dati=["label", "year", "gender", "startDate", "endDate", "countTeams"]),
    "stage": dict(classe="Stage", uri="stage/{tournament_id}/{stageNumber}",
                  dati=["label", "stageNumber", "stageName", "isGroupStage", "isKnockoutStage", "startDate", "endDate"],
                  oggetti={"stageOf": "tournament/{tournament_id}"}),
    "group": dict(classe="Group", uri="group/{tournament_id}/{stageNumber}/{groupLetter}",
                  dati=["label", "groupLetter", "countTeams"],
                  oggetti={"groupOf": "stage/{tournament_id}/{stageNumber}"}),
    "group_standing": dict(classe="GroupStanding",
                           uri="groupstanding/{tournament_id}/{stageNumber}/{groupLetter}/{team_id}",
                           dati=["groupPosition", "points", "advanced"],
                           oggetti={"inGroupStanding": "group/{tournament_id}/{stageNumber}/{groupLetter}",
                                    "participation": "participation/{tournament_id}/{team_id}"}),
    "confederation": dict(classe="Confederation", uri="confederation/{confederation_id}",
                          dati=["label", "confederationCode"], link=True),
    "federation": dict(classe="Federation", uri="federation/{team_id}",
                       dati=["label", "altLabel", "teamCode", "regionName"],
                       oggetti={"memberOf": "confederation/{confederation_id}"}, link=True),
    "federation_predecessor": dict(uri="federation/{team_id}",
                                   oggetti={"predecessorOf": "federation/{successor_id}"}),
    "national_team": dict(classe="NationalTeam", uri="nationalteam/{team_id}/{gender}",
                          dati=["label", "gender"],
                          oggetti={"representsFederation": "federation/{team_id}"}, link=True),
    "participation": dict(classe="Participation", uri="participation/{tournament_id}/{team_id}",
                          dati=["label", "performance", "isHost", "finalPosition", "finalRank", "fifaRanking"],
                          oggetti={"team": "nationalteam/{team_id}/{gender}",
                                   "tournament": "tournament/{tournament_id}",
                                   "kitSponsor": "brand/{kitSponsor_slug}"}),
    "participation_manager": dict(uri="participation/{tournament_id}/{team_id}",
                                  oggetti={"hasManager": "manager/{manager_id}"}),
    # --- Persone e premi
    "player": dict(classe="Player", uri="player/{player_id}",
                   dati=["label", "givenName", "familyName", "birthDate", "isFemale"], link=True),
    "manager": dict(classe="Manager", uri="manager/{manager_id}",
                    dati=["label", "givenName", "familyName", "isFemale", "countryName"], link=True),
    "referee": dict(classe="Referee", uri="referee/{referee_id}",
                    dati=["label", "givenName", "familyName", "isFemale", "countryName"],
                    oggetti={"memberOf": "confederation/{confederation_id}"}, link=True),
    "award": dict(classe="Award", uri="award/{award_id}",
                  dati=["label", "awardDescription", "yearIntroduced"]),
    # --- Stadi e partite
    "stadium": dict(classe="Stadium", uri="stadium/{stadium_id}",
                    dati=["label", "cityName", "countryName", "capacity"], link=True),
    "match": dict(classe="Match", uri="match/{match_id}",
                  dati=["label", "matchDate", "homeScore", "awayScore", "extraTime", "penaltyShootout",
                        "homePenalties", "awayPenalties", "isReplay"],
                  oggetti={"inStage": "stage/{tournament_id}/{stageNumber}",
                           "inGroup": "group/{tournament_id}/{stageNumber}/{groupLetter}",
                           "playedAt": "stadium/{stadium_id}",
                           "homeTeam": "participation/{tournament_id}/{home_team_id}",
                           "awayTeam": "participation/{tournament_id}/{away_team_id}",
                           "refereedBy": "referee/{referee_id}"}),
    # --- Club e marche
    "club": dict(classe="Club", uri="club/{slug}", dati=["label"]),
    "brand": dict(classe="Brand", uri="brand/{slug}", dati=["label"]),
    # --- Convocazioni
    "squad_membership": dict(classe="SquadMembership", uri="membership/{tournament_id}/{team_id}/{player_id}",
                             dati=["shirtNumber", "isCaptain", *DATASET_3_PROPRIETA],
                             oggetti={"player": "player/{player_id}",
                                      "participation": "participation/{tournament_id}/{team_id}",
                                      "position": "wc:position_{position}",
                                      "club": "club/{club_slug}",
                                      "bootBrand": "brand/{bootBrand_slug}"}),
    "membership_award": dict(uri="membership/{tournament_id}/{team_id}/{player_id}",
                             oggetti={"wonAward": "award/{award_id}"}),
    # --- Presenze ed eventi
    "appearance": dict(classe="Appearance", uri="appearance/{match_id}/{player_id}",
                       dati=["shirtNumber", "isStarter", "isSubstitute"],
                       oggetti={"inMatch": "match/{match_id}", "player": "player/{player_id}",
                                "participation": "participation/{tournament_id}/{team_id}",
                                "position": "wc:position_{position}"}),
    "goal": dict(classe="Goal", uri="goal/{goal_id}", dati=[*MINUTO, "isOwnGoal", "isPenalty"],
                  oggetti={**EVENTO, "participation": "participation/{tournament_id}/{player_team_id}",
                           "creditedTo": "participation/{tournament_id}/{team_id}"}),
    "booking": dict(classe="Booking", uri="booking/{booking_id}",
                     dati=[*MINUTO, "isYellowCard", "isRedCard", "isSecondYellow", "isSendingOff"], oggetti=EVENTO),
    "substitution": dict(classe="Substitution", uri="substitution/{substitution_id}",
                          dati=[*MINUTO, "isGoingOff", "isComingOn"], oggetti=EVENTO),
    "penalty_kick": dict(classe="PenaltyKick", uri="penaltykick/{penalty_kick_id}",
                          dati=["isConverted"], oggetti=EVENTO),
    # --- Immagini
    "image": dict(classe="Image", uri="image/{player_id}/{numero}",
                  dati=["filePath", "md5", "fileFormat", "fileSize", "hasDuplicateContent"],
                  oggetti={"depicts": "player/{player_id}"}),
}
print(len(MAPPA), "tabelle in MAPPA")

26 tabelle in MAPPA


In [ ]:
# [Cella 27] Generazione: un file Turtle per ogni voce di MAPPA
# Le tabelle integrate, rilette dai Parquet salvati sopra
TAB = {nome: pd.read_parquet(OUT / f"{nome}.parquet") for nome in MAPPA}
# Lo slug della marca è un dettaglio dell'URI, non un dato della tabella
TAB["participation"]["kitSponsor_slug"] = TAB["participation"]["kitSponsor"].map(slug_nome, na_action="ignore")

conteggi = {}
# Per ogni voce di MAPPA: righe → triple → un file Turtle in data/kg/
for nome, regole in MAPPA.items():
    g = genera(TAB[nome], **regole)                     # **regole: le voci del dizionario come argomenti
    g.serialize(KG / f"{nome}.ttl", format="turtle", encoding="utf-8")
    conteggi[nome] = (len(TAB[nome]), len(g))
    print(f"{nome:23} righe {len(TAB[nome]):5}  triple {len(g):6}")
print("totale triple:", sum(t for _, t in conteggi.values()))

tournament              righe    30  triple    210
stage                   righe   155  triple   1395
group                   righe   159  triple    795
group_standing          righe   626  triple   3756
confederation           righe     6  triple     30
federation              righe    88  triple    702
federation_predecessor  righe     5  triple      5
national_team           righe   121  triple    725
participation           righe   625  triple   4423
participation_manager   righe   637  triple    637
player                  righe 10401  triple  82628
manager                 righe   475  triple   3793
referee                 righe   493  triple   4187
award                   righe     8  triple     32
stadium                 righe   240  triple   1680
match                   righe  1248  triple  17232
club                    righe   291  triple    582
brand                   righe    15  triple     30
squad_membership        righe 13843  triple  74582
membership_award        righe  

In [ ]:
# [Cella 28] Ruoli: i 21 codici del dataset 1 con il loro nome.
ruoli = nuovo_grafo()
for s in onto.subjects(RDF.type, WC.Position):
    for p, o in onto.predicate_objects(s):
        ruoli.add((s, p, o))
ruoli.serialize(KG / "position.ttl", format="turtle", encoding="utf-8")
conteggi["position"] = (len(set(ruoli.subjects(RDF.type, WC.Position))), len(ruoli))
print("position.ttl:", len(ruoli), "triple |", conteggi["position"][0], "ruoli")
print("totale triple:", sum(t for _, t in conteggi.values()))

position.ttl: 84 triple | 21 ruoli
totale triple: 872690


## Import in Neo4j con n10s
Ogni volta si svuota il database e si reimportano tutti i file di `data/kg/`.

In [ ]:
# [Cella 29] Connessione a Neo4j e controllo del plugin n10s
NEO4J_URI = "neo4j://localhost:7687"
NEO4J_USER = "neo4j"
BASE = str(R)                    # prefisso degli URI dei nodi, usato nelle query come $base
# getpass: la password non viene scritta nel notebook
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, getpass("Password Neo4j: ")))
driver.verify_connectivity()

def cypher(query, **parametri):
    # Esegue una query con parametri ($nome) separati dal testo e restituisce un DataFrame
    records, _, _ = driver.execute_query(query, parametri, database_="neo4j")
    return pd.DataFrame([r.data() for r in records])

print(cypher("CALL dbms.components() YIELD name, versions, edition RETURN name, versions[0] AS versione, edition"))
print("procedure n10s:", cypher("SHOW PROCEDURES YIELD name WHERE name STARTS WITH 'n10s' RETURN count(*) AS n")["n"][0])

           name versione     edition
0  Neo4j Kernel  5.26.19  enterprise
procedure n10s: 55


In [ ]:
# [Cella 30] ATTENZIONE: svuota il database; poi vincolo di unicità e configurazione di n10s
cypher("MATCH (n) DETACH DELETE n")
# Un solo nodo per URI: è ciò che collega i file importati separatamente
cypher("CREATE CONSTRAINT n10s_unique_uri IF NOT EXISTS FOR (r:Resource) REQUIRE r.uri IS UNIQUE")
# Solo il nome locale (Participation, finalRank, label, team); modificabile solo a database vuoto
config = cypher('CALL n10s.graphconfig.init({handleVocabUris: "IGNORE"})')
print(config[config["param"].isin(["handleVocabUris", "handleMultival", "handleRDFTypes", "keepCustomDataTypes"])])

                 param      value
0      handleVocabUris     IGNORE
1       handleMultival  OVERWRITE
2       handleRDFTypes     LABELS
4  keepCustomDataTypes      False


In [ ]:
# [Cella 31] Import di tutti i file Turtle e controllo dei tipi
# Le pagine Wikipedia (isPrimaryTopicOf) non si importano: ridondanti rispetto a DBpedia
ESCLUSI = [str(FOAF.isPrimaryTopicOf)]
IMPORT = """
    CALL n10s.rdf.import.inline($rdf, 'Turtle', {predicateExclusionList: $esclusi})
    YIELD terminationStatus, triplesParsed, triplesLoaded, extraInfo
    RETURN terminationStatus, triplesParsed, triplesLoaded, extraInfo"""

risultati = [cypher(IMPORT, rdf=f.read_text(encoding="utf-8"), esclusi=ESCLUSI).assign(file=f.name)
             for f in sorted(KG.glob("*.ttl"))]
imp = pd.concat(risultati, ignore_index=True)[["file", "terminationStatus", "triplesParsed", "triplesLoaded", "extraInfo"]]
print(imp.to_string(index=False))
# triplesParsed conta anche le triple escluse, triplesLoaded no
print("totale lette:", imp["triplesParsed"].sum(), "| caricate:", imp["triplesLoaded"].sum())

TIPI = """
MATCH (t:Tournament {uri: $base + 'tournament/WC-2022'}),
      (p:Participation {uri: $base + 'participation/WC-2022/T-03'})
RETURN valueType(t.year) AS year, valueType(t.startDate) AS startDate,
       valueType(p.isHost) AS isHost, valueType(p.finalRank) AS finalRank, valueType(p.label) AS label
"""
print(cypher(TIPI, base=BASE).T)

                      file terminationStatus  triplesParsed  triplesLoaded extraInfo
            appearance.ttl                OK         219456         219456          
                 award.ttl                OK             32             32          
               booking.ttl                OK          34958          34958          
                 brand.ttl                OK             30             30          
                  club.ttl                OK            582            582          
         confederation.ttl                OK             30             24          
            federation.ttl                OK            702            616          
federation_predecessor.ttl                OK              5              5          
                  goal.ttl                OK          36370          36370          
                 group.ttl                OK            795            795          
        group_standing.ttl                OK           3756      

In [ ]:
# [Cella 32] Anni da testo a intero: xsd:gYear arriva in Neo4j come stringa
print(cypher("MATCH (t:Tournament) SET t.year = toInteger(t.year) RETURN count(t) AS tornei"))
print(cypher("MATCH (a:Award) SET a.yearIntroduced = toInteger(a.yearIntroduced) RETURN count(a) AS premi"))
print(cypher(TIPI, base=BASE).T)

   tornei
0      30
   premi
0      8
                          0
year       INTEGER NOT NULL
startDate     DATE NOT NULL
isHost     BOOLEAN NOT NULL
finalRank  INTEGER NOT NULL
label       STRING NOT NULL
